# Session 4: Automatic differentiation and neural network function approximation

In [Session 3](Session3.ipynb) we built intuition for neural networks: what they are, why they can approximate any function, and how training works in principle. This session we get hands-on in PyTorch. The most important concept to master before writing a physics-informed neural network (PINN) is **automatic differentiation** (autograd), because the entire PINN loss function relies on computing exact derivatives of the neural network output with respect to its inputs.

## 1. PyTorch tensors and the computational graph

A PyTorch `Tensor` is like a NumPy array, but it can optionally track every mathematical operation applied to it in a **computational graph**. When you call `.backward()`, PyTorch traverses this graph in reverse using the chain rule to compute gradients automatically.

Key flags:
- `requires_grad=True` — tells PyTorch to track operations on this tensor.
- `tensor.grad` — stores $\partial L / \partial \text{tensor}$ after `.backward()`.
- `torch.no_grad()` — context manager that disables tracking (useful for inference/evaluation).

Let us start with the simplest possible example:

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# A scalar tensor with gradient tracking
x = torch.tensor(3.0, requires_grad=True)

# Build a simple computation: y = x^2 + 2x + 1
y = x**2 + 2*x + 1

# Backpropagate: dy/dx = 2x + 2
y.backward()

print(f"x = {x.item()}")
print(f"y = x^2 + 2x + 1 = {y.item()}")
print(f"dy/dx (analytical) = 2*3 + 2 = 8")
print(f"dy/dx (autograd)   = {x.grad.item()}")

### Higher-order derivatives with `create_graph=True`

For PINNs we often need second (or higher) order derivatives — e.g., $\partial^2 u / \partial x^2$ in the heat equation. This requires keeping the computational graph alive after the first backward pass via `create_graph=True`.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

# f(x) = sin(x)
f = torch.sin(x)

# First derivative: df/dx = cos(x)
df_dx = torch.autograd.grad(f, x, create_graph=True)[0]

# Second derivative: d^2f/dx^2 = -sin(x)
d2f_dx2 = torch.autograd.grad(df_dx, x)[0]

print(f"x            = {x.item():.4f}")
print(f"sin(x)       = {f.item():.4f}")
print(f"cos(x)       = {torch.cos(x).item():.4f}")
print(f"df/dx        = {df_dx.item():.4f}")
print(f"-sin(x)      = {-torch.sin(x).item():.4f}")
print(f"d^2f/dx^2    = {d2f_dx2.item():.4f}")

## 2. Automatic differentiation vs. finite differences

You might wonder: why not just use finite differences to compute gradients of the network? There are two reasons autograd is vastly superior for PINNs:

1. **Exactness**: Autograd computes derivatives exactly (to floating-point precision) by applying the chain rule through the computation graph. Finite differences introduce truncation error $O(h^2)$ and are sensitive to step size.

2. **Scalability**: Autograd scales to millions of parameters and arbitrary compositions of functions with no extra cost. Finite differences require $2p$ forward passes for $p$ parameters.

The diagram below (conceptually) shows the forward pass building a graph and the backward pass computing all gradients in a single sweep:

```
Forward:  x → [op1] → [op2] → ... → loss
Backward: x ← [∂op1] ← [∂op2] ← ... ← 1
```

For a PINN with 10,000 collocation points, autograd computes the full PDE residual and its gradient in one pass. This is what makes the method computationally feasible.

## 3. Automatic differentiation on batches of points

In PINNs the input is a batch of $(x, t)$ pairs. We need the derivative of the network output $u$ with respect to each input coordinate. This uses `torch.autograd.grad` with `grad_outputs=torch.ones_like(u)`, which computes $\sum_i \partial u_i / \partial x$ — element-wise when the output and input have the same shape.

In [ ]:
# Demonstrate batched autograd: du/dx for u = sin(pi*x) at many points
x_batch = torch.linspace(0, 1, 10).reshape(-1, 1).requires_grad_(True)

u = torch.sin(np.pi * x_batch)

du_dx = torch.autograd.grad(
    u, x_batch,
    grad_outputs=torch.ones_like(u),
    create_graph=True
)[0]

print("x         | du/dx (autograd) | du/dx (analytical: pi*cos(pi*x))")
for xi, di in zip(x_batch.detach().flatten(), du_dx.detach().flatten()):
    analytical = np.pi * torch.cos(np.pi * torch.tensor(xi)).item()
    print(f"  {xi:.3f}   |   {di:.6f}      |   {analytical:.6f}")

## 4. Hands-on: approximating sin(x) with a neural network

Before we write a full PINN next session, let us use a neural network purely as a data-fitting tool: learn to approximate $\sin(x)$ from 500 equally spaced training points. This exercise:

- Gets you comfortable with the PyTorch training loop.
- Shows that neural networks *can* fit smooth functions very well.
- Introduces `requires_grad=True` on input tensors — a habit that carries directly into PINN code.

### 4.1 Define the network

In [ ]:
class Net(nn.Module):
    """Simple MLP: input 1D → hidden layers → output 1D."""
    def __init__(self, layers=[1, 50, 50, 50, 1]):
        super().__init__()
        self.layers = nn.ModuleList()
        for i in range(len(layers) - 2):
            self.layers.append(nn.Linear(layers[i], layers[i+1]))
            self.layers.append(nn.Tanh())
        self.layers.append(nn.Linear(layers[-2], layers[-1]))

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

### 4.2 Generate training data and train

In [ ]:
# Data: 500 points in [-pi, pi]
N = 500
x = torch.linspace(-np.pi, np.pi, N).reshape(-1, 1)
y = torch.sin(x)

# Note: requires_grad=True on inputs — a habit from PINNs
x.requires_grad_(True)

model = Net()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

epochs = 5000
for epoch in range(epochs):
    pred = model(x)
    loss = loss_fn(pred, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 1000 == 0:
        print(f"Epoch {epoch:5d}, Loss: {loss.item():.6f}")

### 4.3 Visualise the fit

In [ ]:
with torch.no_grad():
    x_test = torch.linspace(-np.pi, np.pi, 1000).reshape(-1, 1)
    y_pred = model(x_test)

plt.figure(figsize=(10, 4))
plt.plot(x_test.numpy(), np.sin(x_test.numpy()), 'k-', label='True sin(x)', linewidth=2)
plt.plot(x_test.numpy(), y_pred.numpy(), 'r--', label='NN approximation', linewidth=2)
plt.scatter(x.detach().numpy(), y.numpy(), s=5, alpha=0.3, label='Training points', c='blue')
plt.legend()
plt.title('Neural Network Approximation of sin(x)')
plt.xlabel('x')
plt.ylabel('u')
plt.tight_layout()
plt.show()

### 4.4 Inspecting learned derivatives

One of the key insights we need for PINNs: the derivative of the *neural network* (not the data) can be computed via autograd. Let us verify that our trained model's derivative also approximates $\cos(x) = d(\sin(x))/dx$.

In [ ]:
x_eval = torch.linspace(-np.pi, np.pi, 200).reshape(-1, 1).requires_grad_(True)
u_eval = model(x_eval)

du_dx = torch.autograd.grad(
    u_eval, x_eval,
    grad_outputs=torch.ones_like(u_eval),
    create_graph=False
)[0]

x_np = x_eval.detach().numpy()

plt.figure(figsize=(10, 4))
plt.plot(x_np, np.cos(x_np), 'k-', label='True cos(x)', linewidth=2)
plt.plot(x_np, du_dx.detach().numpy(), 'r--', label='d(NN)/dx via autograd', linewidth=2)
plt.legend()
plt.title('Derivative of the trained NN vs. cos(x)')
plt.xlabel('x')
plt.ylabel("u'")
plt.tight_layout()
plt.show()

The derivative of the trained network closely matches $\cos(x)$, even though we never explicitly trained on derivatives. This is the crucial observation: **if the network fits the function well, its autograd derivatives also approximate the true derivatives**.

In a PINN, we will enforce that these derivatives satisfy the PDE — instead of fitting known data everywhere.

## 5. Summary and what comes next

Key takeaways from this session:
- PyTorch's autograd builds a computational graph and applies the chain rule exactly.
- Setting `requires_grad=True` on input tensors is essential for PINNs.
- `torch.autograd.grad(..., create_graph=True)` enables higher-order derivatives.
- A simple MLP trained on data can approximate smooth functions and their derivatives.

**[Session 5](Session5.ipynb)**: We introduce the formal PINN framework — what it means to enforce a PDE as a loss term, how to sample collocation points, and how to include boundary and initial conditions. We will write our very first PINN from scratch.

**Reading**: Raissi, Perdikaris, Karniadakis (2019) (The original PINN paper — worth skimming the abstract and Section 2.)

## Exercises

1. **Product rule verification**: define $f(x) = x^2 \sin(x)$ as a PyTorch tensor expression. Use autograd to compute $f'(x)$ and $f''(x)$ at several values of $x$, then verify against the analytical results $(2x\sin x + x^2\cos x)$ and $(2\sin x + 4x\cos x - x^2\sin x)$.

2. **2D Laplacian**: create a batch of 2D input points $(x, y)$ with `requires_grad=True` and compute $\nabla^2 u = \partial^2 u/\partial x^2 + \partial^2 u/\partial y^2$ for $u = \sin(\pi x)\sin(\pi y)$ using `torch.autograd.grad`. Verify the result equals $-2\pi^2 u$.

3. **Effect of `create_graph=False`**: attempt to compute the second derivative of a function by calling `torch.autograd.grad` twice but with `create_graph=False` on the first call. What error do you get? Explain why `create_graph=True` is mandatory for higher-order derivatives in PINNs.

4. **Derivative quality vs data density**: train the `Net` from Section 4 on $N = 10, 50, 100, 500$ equally spaced training points. For each model, compute the maximum absolute error in both $u(x)$ and $u'(x)$ over a dense evaluation grid. Plot the error scaling — does derivative quality improve at the same rate as function approximation quality?